## **I have installed the librraies and created the environment here**

In [2]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr poppler-utils

!pip install -q pymupdf
!pip install -q pdf2image
!pip install -q pytesseract
!pip install -q pillow
!pip install -q opencv-python-headless
!pip install -q sentence-transformers
!pip install -q rank-bm25
!pip install -q faiss-cpu
!pip install -q pydantic
!pip install -q rapidfuzz
!pip install -q python-dotenv
!pip install -q openai
!pip install -q pytest

print("Installation completed.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Installation completed.


# **I created the Project Folder here**

In [3]:
from pathlib import Path
import shutil

PROJECT_DIR = Path("/content/legal_doc_ai_assessment")

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

folders = [
    "src/legal_doc_ai",
    "data/sample_docs",
    "outputs/sample_run",
    "tests"
]

for folder in folders:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

print("Project created at:", PROJECT_DIR)

Project created at: /content/legal_doc_ai_assessment


# **I created the sample document in a legal document style**

In [4]:
sample_notice = """
NOTICE TO VACATE

Date: March 12, 2026

From: Northgate Property Management LLC
To: Daniel Harper
Property: 44 West Pine Street, Unit 3B, Seattle, WA 98101

Dear Mr. Harper,

This letter serves as a written notice regarding unpaid rent and repeated late payment issues under the residential lease dated January 1, 2025.

According to our records, rent for February 2026 in the amount of $1,850 remains unpaid. A late fee of $75 was assessed on February 6, 2026. The total balance currently shown is $1,925.

The lease requires monthly rent to be paid on or before the 1st day of each month. The lease also states that unpaid rent may result in further action after written notice.

Please contact the property manager within ten days of this notice to discuss payment or provide proof of payment.

Sincerely,
Maya Chen
Property Manager
Northgate Property Management LLC
"""

sample_lease = """
RESIDENTIAL LEASE AGREEMENT - EXTRACT

Landlord: Northgate Property Management LLC
Tenant: Daniel Harper
Premises: 44 West Pine Street, Unit 3B, Seattle, WA 98101
Lease Start: January 1, 2025
Monthly Rent: $1,850
Due Date: 1st day of each month

Section 4. Rent.
Tenant shall pay monthly rent in the amount of $1,850 on or before the first day of each month.

Section 7. Late Payment.
If rent is not received by the fifth day of the month, a late fee of $75 may be charged.

Section 11. Notices.
Any notice required under this lease must be provided in writing to the tenant at the premises or by another method permitted by applicable law.

Note: Some scan artifacts were present in the original file: "Rent due 1$t day", "late fee $7S". These were normalized after OCR review.
"""

sample_operator_edit = """
Operator edit example:

Original wording:
"The tenant violated the lease and must vacate."

Edited wording:
"The records indicate possible nonpayment and late-payment issues. Further legal review is needed before determining the appropriate next step."

Reusable preference:
Avoid definitive legal conclusions unless the source evidence expressly supports them. Use cautious wording such as "records indicate", "appears", and "requires legal review".
"""

(PROJECT_DIR / "data/sample_docs/notice_to_vacate.txt").write_text(sample_notice, encoding="utf-8")
(PROJECT_DIR / "data/sample_docs/lease_extract_messy.txt").write_text(sample_lease, encoding="utf-8")
(PROJECT_DIR / "data/sample_docs/operator_edit_example.txt").write_text(sample_operator_edit, encoding="utf-8")

print("Sample documents created.")

Sample documents created.


# **I created the Package Initialization File**

In [5]:
(PROJECT_DIR / "src/legal_doc_ai/__init__.py").write_text(
    '"""Legal document AI assessment package."""\n\n__version__ = "0.1.0"\n',
    encoding="utf-8"
)

67

# **Schema Defining**

In [6]:
schemas_code = r'''
from __future__ import annotations

from typing import Dict, List, Optional
from pydantic import BaseModel, Field


class PageRecord(BaseModel):
    doc_id: str
    source_path: str
    page_number: int
    text: str
    extraction_method: str
    ocr_confidence: Optional[float] = None
    warnings: List[str] = Field(default_factory=list)


class StructuredFields(BaseModel):
    doc_id: str
    parties: Dict[str, str] = Field(default_factory=dict)
    dates: Dict[str, str] = Field(default_factory=dict)
    amounts: Dict[str, str] = Field(default_factory=dict)
    property_address: Optional[str] = None
    issues: List[str] = Field(default_factory=list)
    uncertainty_notes: List[str] = Field(default_factory=list)


class Chunk(BaseModel):
    chunk_id: str
    doc_id: str
    source_path: str
    page_number: int
    text: str
    start_char: int
    end_char: int


class RetrievedEvidence(BaseModel):
    chunk_id: str
    doc_id: str
    source_path: str
    page_number: int
    text: str
    score: float
    rank: int
    retrieval_method: str


class DraftResult(BaseModel):
    task: str
    draft: str
    evidence: List[RetrievedEvidence]
    structured_fields: List[StructuredFields]
    backend: str


class EditMemory(BaseModel):
    cautious_legal_language: bool = True
    preferred_phrases: List[str] = Field(
        default_factory=lambda: ["records indicate", "appears", "requires legal review"]
    )
    banned_phrases: List[str] = Field(
        default_factory=lambda: ["must vacate", "violated the lease"]
    )
    learned_notes: List[str] = Field(default_factory=list)


class EvaluationReport(BaseModel):
    retrieval_recall_at_k: float
    grounding_coverage: float
    unsupported_claim_control: bool
    edit_learning_applied: bool
    notes: List[str] = Field(default_factory=list)
'''

(PROJECT_DIR / "src/legal_doc_ai/schemas.py").write_text(schemas_code, encoding="utf-8")

print("schemas.py created.")

schemas.py created.


# **Utility Function Creation**

In [7]:
utils_code = r'''
from __future__ import annotations

import json
import os
import random
from pathlib import Path
from typing import Any, Iterable

import numpy as np


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def write_json(path: str | Path, data: Any) -> None:
    Path(path).write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )


def read_json(path: str | Path) -> Any:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_jsonl(path: str | Path, records: Iterable[Any]) -> None:
    with Path(path).open("w", encoding="utf-8") as f:
        for record in records:
            if hasattr(record, "model_dump"):
                record = record.model_dump()
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def stable_doc_id(path: str | Path) -> str:
    path = Path(path)
    value = path.stem.lower().replace(" ", "_").replace("-", "_")
    return "".join(ch for ch in value if ch.isalnum() or ch == "_")
'''

(PROJECT_DIR / "src/legal_doc_ai/utils.py").write_text(utils_code, encoding="utf-8")

print("utils.py created.")

utils.py created.


# **Config File Creation**

In [8]:
config_code = r'''
from __future__ import annotations

import os
from dataclasses import dataclass


@dataclass(frozen=True)
class Settings:
    generation_backend: str = os.getenv("GENERATION_BACKEND", "template")
    openai_model: str = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    embedding_model: str = os.getenv(
        "EMBEDDING_MODEL",
        "sentence-transformers/all-MiniLM-L6-v2"
    )
    random_seed: int = int(os.getenv("RANDOM_SEED", "42"))
    chunk_size_chars: int = int(os.getenv("CHUNK_SIZE_CHARS", "900"))
    chunk_overlap_chars: int = int(os.getenv("CHUNK_OVERLAP_CHARS", "150"))
    top_k: int = int(os.getenv("TOP_K", "6"))
'''

(PROJECT_DIR / "src/legal_doc_ai/config.py").write_text(config_code, encoding="utf-8")

print("config.py created.")

config.py created.


# **Document processor with OCR support**

In [9]:
document_processor_code = r'''
from __future__ import annotations

import re
from pathlib import Path
from typing import List, Tuple

import cv2
import fitz
import numpy as np
import pytesseract
from pdf2image import convert_from_path
from PIL import Image

from .schemas import PageRecord, StructuredFields
from .utils import stable_doc_id


SUPPORTED_EXTENSIONS = {
    ".txt", ".md", ".pdf", ".png", ".jpg", ".jpeg", ".tif", ".tiff"
}


class DocumentProcessor:
    def __init__(self, min_pdf_text_chars: int = 40):
        self.min_pdf_text_chars = min_pdf_text_chars

    def process_directory(
        self,
        input_dir: str | Path
    ) -> Tuple[List[PageRecord], List[StructuredFields]]:
        input_dir = Path(input_dir)

        if not input_dir.exists():
            raise FileNotFoundError(f"Input directory does not exist: {input_dir}")

        files = [
            file for file in input_dir.rglob("*")
            if file.is_file() and file.suffix.lower() in SUPPORTED_EXTENSIONS
        ]

        if not files:
            raise ValueError(f"No supported documents found in {input_dir}")

        all_pages = []
        all_structured_fields = []

        for file_path in sorted(files):
            pages = self.process_file(file_path)
            all_pages.extend(pages)

            combined_text = "\n".join(page.text for page in pages)
            structured = self.extract_structured_fields(
                doc_id=stable_doc_id(file_path),
                text=combined_text
            )
            all_structured_fields.append(structured)

        return all_pages, all_structured_fields

    def process_file(self, file_path: str | Path) -> List[PageRecord]:
        file_path = Path(file_path)
        suffix = file_path.suffix.lower()

        if suffix in {".txt", ".md"}:
            return self._process_text(file_path)

        if suffix == ".pdf":
            return self._process_pdf(file_path)

        if suffix in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}:
            return self._process_image(file_path)

        raise ValueError(f"Unsupported file type: {file_path}")

    def _process_text(self, file_path: Path) -> List[PageRecord]:
        text = file_path.read_text(encoding="utf-8", errors="replace")

        return [
            PageRecord(
                doc_id=stable_doc_id(file_path),
                source_path=str(file_path),
                page_number=1,
                text=self._normalize_text(text),
                extraction_method="text",
                ocr_confidence=None,
                warnings=[]
            )
        ]

    def _process_pdf(self, file_path: Path) -> List[PageRecord]:
        doc_id = stable_doc_id(file_path)
        records = []

        with fitz.open(file_path) as pdf:
            for page_index, page in enumerate(pdf, start=1):
                embedded_text = self._normalize_text(page.get_text("text") or "")

                if len(embedded_text.strip()) >= self.min_pdf_text_chars:
                    records.append(
                        PageRecord(
                            doc_id=doc_id,
                            source_path=str(file_path),
                            page_number=page_index,
                            text=embedded_text,
                            extraction_method="pdf_text",
                            ocr_confidence=None,
                            warnings=[]
                        )
                    )
                else:
                    ocr_text, confidence = self._ocr_pdf_page(file_path, page_index)

                    warnings = []
                    if confidence is not None and confidence < 55:
                        warnings.append(
                            "Low OCR confidence; this page may be partially unclear."
                        )

                    records.append(
                        PageRecord(
                            doc_id=doc_id,
                            source_path=str(file_path),
                            page_number=page_index,
                            text=self._normalize_text(ocr_text),
                            extraction_method="ocr_pdf_page",
                            ocr_confidence=confidence,
                            warnings=warnings
                        )
                    )

        return records

    def _process_image(self, file_path: Path) -> List[PageRecord]:
        text, confidence = self._ocr_image(file_path)

        warnings = []
        if confidence is not None and confidence < 55:
            warnings.append("Low OCR confidence; this image may be partially unclear.")

        return [
            PageRecord(
                doc_id=stable_doc_id(file_path),
                source_path=str(file_path),
                page_number=1,
                text=self._normalize_text(text),
                extraction_method="ocr_image",
                ocr_confidence=confidence,
                warnings=warnings
            )
        ]

    def _ocr_pdf_page(
        self,
        file_path: Path,
        page_number: int
    ) -> tuple[str, float | None]:
        images = convert_from_path(
            str(file_path),
            dpi=250,
            first_page=page_number,
            last_page=page_number
        )

        if not images:
            return "", None

        return self._ocr_pil_image(images[0])

    def _ocr_image(self, file_path: Path) -> tuple[str, float | None]:
        image = Image.open(file_path).convert("RGB")
        return self._ocr_pil_image(image)

    def _ocr_pil_image(self, image: Image.Image) -> tuple[str, float | None]:
        processed = self._preprocess_image_for_ocr(image)

        data = pytesseract.image_to_data(
            processed,
            output_type=pytesseract.Output.DICT,
            config="--psm 6"
        )

        words = []
        confidences = []

        for word, confidence in zip(data.get("text", []), data.get("conf", [])):
            word = word.strip()

            try:
                confidence_value = float(confidence)
            except ValueError:
                confidence_value = -1

            if word:
                words.append(word)

                if confidence_value >= 0:
                    confidences.append(confidence_value)

        average_confidence = (
            float(np.mean(confidences)) if confidences else None
        )

        return " ".join(words), average_confidence

    def _preprocess_image_for_ocr(self, image: Image.Image) -> Image.Image:
        array = np.array(image)
        gray = cv2.cvtColor(array, cv2.COLOR_RGB2GRAY)
        denoised = cv2.fastNlMeansDenoising(gray, None, 20, 7, 21)

        thresholded = cv2.threshold(
            denoised,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )[1]

        return Image.fromarray(thresholded)

    def extract_structured_fields(
        self,
        doc_id: str,
        text: str
    ) -> StructuredFields:
        fields = StructuredFields(doc_id=doc_id)

        patterns = {
            "landlord": r"(?:Landlord|From):\s*([^\n]+)",
            "tenant": r"(?:Tenant|To):\s*([^\n]+)",
            "property": r"(?:Property|Premises):\s*([^\n]+)",
            "lease_start": r"(?:Lease Start|lease dated):\s*([^\n\.]+)",
            "notice_date": r"(?:Date):\s*([^\n]+)",
            "monthly_rent": r"(?:Monthly Rent|rent in the amount of)[:\s]*\$?([0-9,]+)",
            "late_fee": r"(?:late fee(?: of)?|Late Payment).*?\$([0-9,]+)",
            "balance": r"(?:total balance.*?|balance currently shown is)\s*\$?([0-9,]+)",
        }

        for key, pattern in patterns.items():
            match = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)

            if not match:
                continue

            value = match.group(1).strip(" .;")

            if key in {"landlord", "tenant"}:
                fields.parties[key] = value
            elif key in {"lease_start", "notice_date"}:
                fields.dates[key] = value
            elif key in {"monthly_rent", "late_fee", "balance"}:
                fields.amounts[key] = "$" + value if not value.startswith("$") else value
            elif key == "property":
                fields.property_address = value

        issue_patterns = [
            ("unpaid rent", r"unpaid rent|rent .* remains unpaid|nonpayment"),
            ("late payment", r"late payment|late fee|not received by the fifth"),
            ("written notice", r"written notice|notice required|provided in writing"),
        ]

        for issue_name, pattern in issue_patterns:
            if re.search(pattern, text, flags=re.IGNORECASE):
                fields.issues.append(issue_name)

        required_fields = [
            ("tenant", fields.parties.get("tenant")),
            ("landlord", fields.parties.get("landlord")),
            ("property_address", fields.property_address),
        ]

        for field_name, value in required_fields:
            if not value:
                fields.uncertainty_notes.append(
                    f"{field_name} not confidently extracted"
                )

        return fields

    def _normalize_text(self, text: str) -> str:
        text = text.replace("\x0c", "\n")
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()
'''

(PROJECT_DIR / "src/legal_doc_ai/document_processor.py").write_text(
    document_processor_code,
    encoding="utf-8"
)

print("document_processor.py created.")

document_processor.py created.


# **Creating logics to retrieve in chunks**

In [10]:
chunking_code = r'''
from __future__ import annotations

from typing import List

from .schemas import Chunk, PageRecord


class TextChunker:
    def __init__(
        self,
        chunk_size_chars: int = 900,
        overlap_chars: int = 150
    ):
        if overlap_chars >= chunk_size_chars:
            raise ValueError("overlap_chars must be smaller than chunk_size_chars")

        self.chunk_size_chars = chunk_size_chars
        self.overlap_chars = overlap_chars

    def chunk_pages(self, pages: list[PageRecord]) -> List[Chunk]:
        chunks = []

        for page in pages:
            text = page.text.strip()

            if not text:
                continue

            start = 0
            chunk_index = 0

            while start < len(text):
                end = min(start + self.chunk_size_chars, len(text))
                window = text[start:end]

                if end < len(text):
                    sentence_end = max(window.rfind("."), window.rfind("\n"))

                    if sentence_end > int(self.chunk_size_chars * 0.55):
                        end = start + sentence_end + 1
                        window = text[start:end]

                chunk_id = f"{page.doc_id}:p{page.page_number}:c{chunk_index}"

                chunks.append(
                    Chunk(
                        chunk_id=chunk_id,
                        doc_id=page.doc_id,
                        source_path=page.source_path,
                        page_number=page.page_number,
                        text=window.strip(),
                        start_char=start,
                        end_char=end
                    )
                )

                if end == len(text):
                    break

                start = max(0, end - self.overlap_chars)
                chunk_index += 1

        return chunks
'''

(PROJECT_DIR / "src/legal_doc_ai/chunking.py").write_text(chunking_code, encoding="utf-8")

print("chunking.py created.")

chunking.py created.


# **Creating a hybrid retriever**

In [11]:
retriever_code = r'''
from __future__ import annotations

import re
from typing import List

import faiss
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

from .schemas import Chunk, RetrievedEvidence


def simple_tokenize(text: str) -> list[str]:
    return re.findall(r"[a-zA-Z0-9$,.]+", text.lower())


class HybridRetriever:
    def __init__(
        self,
        embedding_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    ):
        self.embedding_model_name = embedding_model_name
        self.embedding_model = None
        self.chunks = []
        self.bm25 = None
        self.index = None
        self.embeddings = None

    def fit(self, chunks: list[Chunk]) -> None:
        if not chunks:
            raise ValueError("Cannot fit retriever with no chunks.")

        self.chunks = chunks

        tokenized_corpus = [simple_tokenize(chunk.text) for chunk in chunks]
        self.bm25 = BM25Okapi(tokenized_corpus)

        self.embedding_model = SentenceTransformer(self.embedding_model_name)

        embeddings = self.embedding_model.encode(
            [chunk.text for chunk in chunks],
            normalize_embeddings=True,
            show_progress_bar=False
        )

        self.embeddings = np.asarray(embeddings, dtype="float32")

        self.index = faiss.IndexFlatIP(self.embeddings.shape[1])
        self.index.add(self.embeddings)

    def search(self, query: str, top_k: int = 6) -> List[RetrievedEvidence]:
        if self.bm25 is None or self.index is None or self.embedding_model is None:
            raise RuntimeError("Retriever must be fitted before search.")

        top_k = min(top_k, len(self.chunks))

        query_tokens = simple_tokenize(query)

        bm25_scores = self.bm25.get_scores(query_tokens)
        bm25_order = np.argsort(bm25_scores)[::-1][:top_k * 2]

        query_embedding = self.embedding_model.encode(
            [query],
            normalize_embeddings=True,
            show_progress_bar=False
        )

        query_embedding = np.asarray(query_embedding, dtype="float32")
        vector_scores, vector_indices = self.index.search(query_embedding, top_k * 2)

        candidates = {}

        for rank, index in enumerate(bm25_order):
            candidates[int(index)] = {
                "score": float(bm25_scores[index]) + 1.0 / (rank + 1),
                "method": "bm25"
            }

        for rank, (index, score) in enumerate(
            zip(vector_indices[0], vector_scores[0])
        ):
            index = int(index)

            existing = candidates.get(
                index,
                {
                    "score": 0.0,
                    "method": "vector"
                }
            )

            existing["score"] = (
                float(existing["score"]) + float(score) + 1.0 / (rank + 1)
            )

            existing["method"] = (
                "hybrid" if existing["method"] != "vector" else "vector"
            )

            candidates[index] = existing

        ranked_candidates = sorted(
            candidates.items(),
            key=lambda item: item[1]["score"],
            reverse=True
        )[:top_k]

        evidence = []

        for rank, (index, metadata) in enumerate(ranked_candidates, start=1):
            chunk = self.chunks[index]

            evidence.append(
                RetrievedEvidence(
                    chunk_id=chunk.chunk_id,
                    doc_id=chunk.doc_id,
                    source_path=chunk.source_path,
                    page_number=chunk.page_number,
                    text=chunk.text,
                    score=float(metadata["score"]),
                    rank=rank,
                    retrieval_method=metadata["method"]
                )
            )

        return evidence
'''

(PROJECT_DIR / "src/legal_doc_ai/retriever.py").write_text(retriever_code, encoding="utf-8")

print("retriever.py created.")

retriever.py created.


# **Creating grounded draft generator**

In [12]:
generator_code = r'''
from __future__ import annotations

import os
from typing import List

from .schemas import DraftResult, RetrievedEvidence, StructuredFields


class GroundedDraftGenerator:
    def __init__(
        self,
        backend: str = "template",
        openai_model: str = "gpt-4o-mini"
    ):
        self.backend = backend
        self.openai_model = openai_model

    def generate(
        self,
        task: str,
        evidence: List[RetrievedEvidence],
        structured_fields: List[StructuredFields]
    ) -> DraftResult:
        if self.backend == "openai" and os.getenv("OPENAI_API_KEY"):
            draft = self._generate_openai(task, evidence, structured_fields)
            backend = "openai"
        else:
            draft = self._generate_template(task, evidence, structured_fields)
            backend = "template"

        return DraftResult(
            task=task,
            draft=draft,
            evidence=evidence,
            structured_fields=structured_fields,
            backend=backend
        )

    def _generate_template(
        self,
        task: str,
        evidence: List[RetrievedEvidence],
        structured_fields: List[StructuredFields]
    ) -> str:
        merged = self._merge_fields(structured_fields)

        tenant = merged["parties"].get("tenant", "Unclear from provided documents")
        landlord = merged["parties"].get("landlord", "Unclear from provided documents")
        property_address = merged.get("property_address") or "Unclear from provided documents"

        monthly_rent = merged["amounts"].get("monthly_rent", "Unclear")
        late_fee = merged["amounts"].get("late_fee", "Unclear")
        balance = merged["amounts"].get("balance", "Unclear")

        notice_date = merged["dates"].get("notice_date", "Unclear")
        lease_start = merged["dates"].get("lease_start", "Unclear")

        issues = sorted(set(merged["issues"]))
        issue_text = ", ".join(issues) if issues else "Unclear"

        evidence_lines = []

        for item in evidence:
            preview = item.text[:350].replace("\n", " ").strip()
            evidence_lines.append(
                f"- [{item.chunk_id}] {item.source_path}, page {item.page_number}: {preview}..."
            )

        evidence_reference_text = ", ".join(
            f"[{item.chunk_id}]" for item in evidence[:3]
        )

        draft = f"""# Notice-Related Case Fact Summary

## Drafting task
{task}

## Short answer
The records indicate possible notice-related issues involving unpaid rent, late payment, and written notice requirements. This summary is limited to the retrieved source evidence and does not make a final legal conclusion.

## Key extracted facts
- Tenant: {tenant}
- Landlord / property manager: {landlord}
- Property: {property_address}
- Notice date: {notice_date}
- Lease start or lease date: {lease_start}
- Monthly rent: {monthly_rent}
- Late fee: {late_fee}
- Stated balance: {balance}
- Main issues found: {issue_text}

## Source-grounded summary
The available records indicate that the tenant and landlord/property manager relationship concerns the property at {property_address}. The documents identify rent-related obligations, including monthly rent of {monthly_rent} and a late-fee provision of {late_fee}. The notice document states that a balance of {balance} was shown for the relevant period. These points are supported by {evidence_reference_text}.

The evidence also indicates that notices under the lease must be provided in writing. Based only on the current documents, the safest conclusion is that the matter appears to involve possible nonpayment, late-payment issues, and written notice requirements. Further legal review is needed before determining the appropriate next step.

## Unsupported or unclear points
- Whether the notice fully complies with local law is not determined from these documents.
- Whether payment was later made is not shown in the retrieved evidence.
- Whether the tenant received the notice is not confirmed by the retrieved evidence.
- No final legal conclusion is made.

## Evidence used
{chr(10).join(evidence_lines)}
"""

        return draft

    def _generate_openai(
        self,
        task: str,
        evidence: List[RetrievedEvidence],
        structured_fields: List[StructuredFields]
    ) -> str:
        from openai import OpenAI

        client = OpenAI()

        evidence_text = "\n\n".join(
            f"[{item.chunk_id}] Source: {item.source_path}, page {item.page_number}\n{item.text}"
            for item in evidence
        )

        structured_text = "\n".join(
            item.model_dump_json(indent=2)
            for item in structured_fields
        )

        prompt = f"""
You are drafting a first-pass internal legal-style notice/case fact summary.

Rules:
1. Use only the provided evidence and structured fields.
2. Do not invent facts.
3. If something is unsupported, write it under "Unsupported or unclear points".
4. Attach evidence IDs like [doc:p1:c0] to material claims.
5. Use cautious wording.
6. Do not make definitive legal conclusions.

Task:
{task}

Structured fields:
{structured_text}

Evidence:
{evidence_text}

Return a clean markdown draft.
"""

        response = client.chat.completions.create(
            model=self.openai_model,
            messages=[
                {
                    "role": "system",
                    "content": "You produce grounded legal-style drafts from retrieved evidence only."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.1
        )

        return response.choices[0].message.content or ""

    def _merge_fields(
        self,
        structured_fields: List[StructuredFields]
    ) -> dict:
        merged = {
            "parties": {},
            "dates": {},
            "amounts": {},
            "property_address": None,
            "issues": [],
            "uncertainty_notes": []
        }

        for item in structured_fields:
            merged["parties"].update(
                {key: value for key, value in item.parties.items() if value}
            )
            merged["dates"].update(
                {key: value for key, value in item.dates.items() if value}
            )
            merged["amounts"].update(
                {key: value for key, value in item.amounts.items() if value}
            )

            if not merged["property_address"] and item.property_address:
                merged["property_address"] = item.property_address

            merged["issues"].extend(item.issues)
            merged["uncertainty_notes"].extend(item.uncertainty_notes)

        return merged
'''

(PROJECT_DIR / "src/legal_doc_ai/generator.py").write_text(generator_code, encoding="utf-8")

print("generator.py created.")

generator.py created.


# **Creating edit learner**

In [13]:
edit_learner_code = r'''
from __future__ import annotations

import difflib
from pathlib import Path

from .schemas import EditMemory
from .utils import read_json, write_json


class EditLearner:
    def __init__(self, memory_path: str | Path):
        self.memory_path = Path(memory_path)

        if self.memory_path.exists():
            self.memory = EditMemory(**read_json(self.memory_path))
        else:
            self.memory = EditMemory()

    def learn_from_edit(
        self,
        default_draft: str,
        edited_draft: str
    ) -> EditMemory:
        diff = list(
            difflib.unified_diff(
                default_draft.splitlines(),
                edited_draft.splitlines(),
                lineterm=""
            )
        )

        removed_text = "\n".join(
            line[1:] for line in diff
            if line.startswith("-") and not line.startswith("---")
        )

        added_text = "\n".join(
            line[1:] for line in diff
            if line.startswith("+") and not line.startswith("+++")
        )

        if (
            "must vacate" in removed_text.lower()
            or "violated the lease" in removed_text.lower()
        ):
            self.memory.cautious_legal_language = True

            note = (
                "Operator softened definitive legal conclusions into cautious "
                "source-grounded language."
            )

            if note not in self.memory.learned_notes:
                self.memory.learned_notes.append(note)

        for phrase in [
            "records indicate",
            "appears",
            "requires legal review",
            "possible nonpayment"
        ]:
            if phrase in added_text.lower() and phrase not in self.memory.preferred_phrases:
                self.memory.preferred_phrases.append(phrase)

        for phrase in [
            "must vacate",
            "violated the lease",
            "clearly liable"
        ]:
            if phrase in removed_text.lower() and phrase not in self.memory.banned_phrases:
                self.memory.banned_phrases.append(phrase)

        self.save()
        return self.memory

    def apply(self, draft: str) -> str:
        improved_draft = draft

        replacements = {
            "The tenant violated the lease and must vacate.": (
                "The records indicate possible nonpayment and late-payment issues. "
                "Further legal review is needed before determining the appropriate next step."
            ),
            "must vacate": "may require further action after legal review",
            "violated the lease": "appears to have unresolved lease-related issues",
            "clearly liable": "potentially responsible, subject to legal review"
        }

        if self.memory.cautious_legal_language:
            for old_text, new_text in replacements.items():
                improved_draft = improved_draft.replace(old_text, new_text)

        if "Operator-edit preferences applied" not in improved_draft:
            improved_draft += "\n\n## Operator-edit preferences applied\n"
            improved_draft += "- Used cautious legal wording where the source evidence does not support a definitive conclusion.\n"
            improved_draft += "- Preserved grounding by keeping evidence references attached to key claims.\n"

        return improved_draft

    def save(self) -> None:
        write_json(self.memory_path, self.memory.model_dump())
'''

(PROJECT_DIR / "src/legal_doc_ai/edit_learner.py").write_text(
    edit_learner_code,
    encoding="utf-8"
)

print("edit_learner.py created.")

edit_learner.py created.


# **Creating evaluator**

In [14]:
evaluator_code = r'''
from __future__ import annotations

from rapidfuzz import fuzz

from .schemas import EvaluationReport, RetrievedEvidence


class Evaluator:
    def evaluate(
        self,
        evidence: list[RetrievedEvidence],
        draft: str,
        improved_draft: str
    ) -> EvaluationReport:
        gold_facts = [
            "monthly rent",
            "$1,850",
            "late fee",
            "$75",
            "written notice",
            "unpaid rent"
        ]

        evidence_blob = "\n".join(item.text.lower() for item in evidence)

        hits = 0

        for fact in gold_facts:
            if (
                fact.lower() in evidence_blob
                or fuzz.partial_ratio(fact.lower(), evidence_blob) > 85
            ):
                hits += 1

        retrieval_recall_at_k = hits / len(gold_facts)

        evidence_ids = [item.chunk_id for item in evidence]
        referenced_count = sum(
            1 for evidence_id in evidence_ids
            if evidence_id in draft
        )

        grounding_coverage = referenced_count / max(1, len(evidence_ids))

        unsupported_claim_control = (
            "Unsupported or unclear points" in draft
            and "not confirmed" in draft
            and "No final legal conclusion" in draft
        )

        edit_learning_applied = (
            "Operator-edit preferences applied" in improved_draft
            and "Further legal review" in improved_draft
        )

        notes = [
            "Synthetic gold facts are used for a small reviewer-friendly evaluation.",
            "Grounding coverage checks whether retrieved evidence IDs are visible in the draft.",
            "Unsupported claim control checks whether missing facts are explicitly marked unclear.",
            "Edit learning checks whether operator preferences are applied to later drafts."
        ]

        return EvaluationReport(
            retrieval_recall_at_k=round(retrieval_recall_at_k, 3),
            grounding_coverage=round(grounding_coverage, 3),
            unsupported_claim_control=unsupported_claim_control,
            edit_learning_applied=edit_learning_applied,
            notes=notes
        )
'''

(PROJECT_DIR / "src/legal_doc_ai/evaluator.py").write_text(evaluator_code, encoding="utf-8")

print("evaluator.py created.")

evaluator.py created.


# **In most cases models provides Hallucionated Outcomes. This is very important to mitigate. As I have a research paper on this approach I know how hallicination can bring chaos.**

**So, I added the hallucination removal feature as an extra feature to add a clarity and transparency on the performance.** I call it a Hallucination Guard.

In [15]:
hallucination_guard_code = r'''
from __future__ import annotations

import re
from dataclasses import dataclass
from typing import List, Dict, Any

from .schemas import RetrievedEvidence, StructuredFields


@dataclass
class ClaimCheck:
    claim: str
    support_score: float
    supported: bool
    best_evidence_id: str | None
    reason: str


class HallucinationGuard:
    """
    A lightweight evidence-grounding guard.

    Purpose:
    - Detect unsupported claims in the generated draft.
    - Remove or quarantine hallucinated sentences.
    - Save an inspectable hallucination report.
    - Keep only claims that are supported by retrieved evidence.

    This is intentionally transparent and reproducible.
    It does not rely on another LLM to judge hallucination.
    """

    def __init__(
        self,
        min_support_score: float = 0.22,
        min_claim_tokens: int = 5
    ):
        self.min_support_score = min_support_score
        self.min_claim_tokens = min_claim_tokens

        self.high_risk_legal_phrases = [
            "must vacate",
            "violated the lease",
            "clearly liable",
            "legally required",
            "fully compliant",
            "valid notice",
            "invalid notice",
            "breached the lease",
            "eviction is proper",
            "eviction is valid",
            "tenant is liable",
            "landlord is liable"
        ]

        self.safe_uncertainty_phrases = [
            "unclear",
            "not confirmed",
            "not determined",
            "not shown",
            "requires legal review",
            "possible",
            "appears",
            "records indicate",
            "based only on"
        ]

    def guard_draft(
        self,
        draft: str,
        evidence: List[RetrievedEvidence],
        structured_fields: List[StructuredFields]
    ) -> tuple[str, Dict[str, Any]]:
        evidence_items = self._prepare_evidence(evidence, structured_fields)

        sections = self._split_markdown_sections(draft)

        guarded_sections = []
        removed_claims = []
        checked_claims = []

        for heading, body in sections:
            if self._is_exempt_section(heading):
                guarded_sections.append((heading, body))
                continue

            guarded_body, section_removed, section_checked = self._guard_section(
                body=body,
                evidence_items=evidence_items
            )

            guarded_sections.append((heading, guarded_body))
            removed_claims.extend(section_removed)
            checked_claims.extend(section_checked)

        guarded_draft = self._rebuild_sections(guarded_sections)

        if removed_claims:
            guarded_draft += "\n\n## Hallucination guard notes\n"
            guarded_draft += (
                "The following unsupported or high-risk claims were removed "
                "or quarantined because they were not sufficiently supported "
                "by the retrieved evidence:\n"
            )

            for item in removed_claims:
                guarded_draft += f"- {item['claim']}\n"

        report = {
            "guard_enabled": True,
            "min_support_score": self.min_support_score,
            "total_claims_checked": len(checked_claims),
            "supported_claims": sum(1 for item in checked_claims if item["supported"]),
            "removed_or_quarantined_claims": len(removed_claims),
            "removed_claims": removed_claims,
            "claim_checks": checked_claims
        }

        return guarded_draft.strip() + "\n", report

    def _prepare_evidence(
        self,
        evidence: List[RetrievedEvidence],
        structured_fields: List[StructuredFields]
    ) -> list[dict]:
        evidence_items = []

        for item in evidence:
            evidence_items.append(
                {
                    "id": item.chunk_id,
                    "text": item.text,
                    "tokens": self._tokenize(item.text)
                }
            )

        structured_text_parts = []

        for field in structured_fields:
            structured_text_parts.append(field.model_dump_json())

        structured_text = "\n".join(structured_text_parts)

        if structured_text.strip():
            evidence_items.append(
                {
                    "id": "structured_fields",
                    "text": structured_text,
                    "tokens": self._tokenize(structured_text)
                }
            )

        return evidence_items

    def _split_markdown_sections(self, draft: str) -> list[tuple[str, str]]:
        lines = draft.splitlines()

        sections = []
        current_heading = ""
        current_body = []

        for line in lines:
            if line.startswith("#"):
                if current_heading or current_body:
                    sections.append((current_heading, "\n".join(current_body).strip()))
                current_heading = line.strip()
                current_body = []
            else:
                current_body.append(line)

        if current_heading or current_body:
            sections.append((current_heading, "\n".join(current_body).strip()))

        return sections

    def _rebuild_sections(self, sections: list[tuple[str, str]]) -> str:
        parts = []

        for heading, body in sections:
            if heading:
                parts.append(heading)
            if body:
                parts.append(body)

        return "\n\n".join(parts)

    def _is_exempt_section(self, heading: str) -> bool:
        heading_lower = heading.lower()

        exempt_keywords = [
            "unsupported or unclear",
            "evidence used",
            "hallucination guard",
            "drafting task"
        ]

        return any(keyword in heading_lower for keyword in exempt_keywords)

    def _guard_section(
        self,
        body: str,
        evidence_items: list[dict]
    ) -> tuple[str, list[dict], list[dict]]:
        lines = body.splitlines()

        guarded_lines = []
        removed_claims = []
        checked_claims = []

        for line in lines:
            stripped = line.strip()

            if not stripped:
                guarded_lines.append(line)
                continue

            if stripped.startswith("- "):
                claim_text = stripped[2:].strip()
                prefix = "- "
            else:
                claim_text = stripped
                prefix = ""

            if not self._is_material_claim(claim_text):
                guarded_lines.append(line)
                continue

            claim_check = self._check_claim(claim_text, evidence_items)

            checked_claims.append(
                {
                    "claim": claim_check.claim,
                    "support_score": round(claim_check.support_score, 3),
                    "supported": claim_check.supported,
                    "best_evidence_id": claim_check.best_evidence_id,
                    "reason": claim_check.reason
                }
            )

            if claim_check.supported:
                guarded_lines.append(line)
            else:
                removed_claims.append(
                    {
                        "claim": claim_check.claim,
                        "support_score": round(claim_check.support_score, 3),
                        "best_evidence_id": claim_check.best_evidence_id,
                        "reason": claim_check.reason
                    }
                )

        guarded_body = "\n".join(guarded_lines).strip()

        return guarded_body, removed_claims, checked_claims

    def _is_material_claim(self, text: str) -> bool:
        text_lower = text.lower()
        tokens = self._tokenize(text)

        if len(tokens) < self.min_claim_tokens:
            return False

        if any(phrase in text_lower for phrase in self.safe_uncertainty_phrases):
            return True

        has_amount = bool(re.search(r"\$[0-9,]+", text))
        has_date = bool(
            re.search(
                r"\b(?:january|february|march|april|may|june|july|august|"
                r"september|october|november|december)\b",
                text_lower
            )
        )
        has_legal_keyword = any(
            word in text_lower
            for word in [
                "tenant",
                "landlord",
                "notice",
                "rent",
                "lease",
                "fee",
                "balance",
                "property",
                "payment",
                "unpaid",
                "received",
                "complies",
                "legal",
                "vacate",
                "liable"
            ]
        )

        return has_amount or has_date or has_legal_keyword

    def _check_claim(
        self,
        claim: str,
        evidence_items: list[dict]
    ) -> ClaimCheck:
        claim_lower = claim.lower()

        high_risk = any(
            phrase in claim_lower
            for phrase in self.high_risk_legal_phrases
        )

        best_score = 0.0
        best_evidence_id = None

        claim_tokens = self._tokenize(claim)

        for evidence in evidence_items:
            score = self._support_score(claim_tokens, evidence["tokens"])

            if score > best_score:
                best_score = score
                best_evidence_id = evidence["id"]

        if high_risk and best_score < 0.65:
            return ClaimCheck(
                claim=claim,
                support_score=best_score,
                supported=False,
                best_evidence_id=best_evidence_id,
                reason="High-risk legal conclusion without strong evidence support."
            )

        if best_score >= self.min_support_score:
            return ClaimCheck(
                claim=claim,
                support_score=best_score,
                supported=True,
                best_evidence_id=best_evidence_id,
                reason="Claim has enough lexical overlap with retrieved evidence."
            )

        return ClaimCheck(
            claim=claim,
            support_score=best_score,
            supported=False,
            best_evidence_id=best_evidence_id,
            reason="Claim is not sufficiently supported by retrieved evidence."
        )

    def _support_score(
        self,
        claim_tokens: set[str],
        evidence_tokens: set[str]
    ) -> float:
        if not claim_tokens:
            return 0.0

        overlap = claim_tokens.intersection(evidence_tokens)

        important_tokens = {
            token for token in claim_tokens
            if len(token) > 3 or token.startswith("$") or token.isdigit()
        }

        if important_tokens:
            important_overlap = important_tokens.intersection(evidence_tokens)
            return len(important_overlap) / len(important_tokens)

        return len(overlap) / len(claim_tokens)

    def _tokenize(self, text: str) -> set[str]:
        stopwords = {
            "the", "and", "or", "of", "to", "a", "an", "is", "are", "was",
            "were", "in", "on", "for", "with", "by", "from", "that", "this",
            "as", "at", "be", "been", "it", "its", "under", "only", "current"
        }

        tokens = re.findall(r"\$?[a-zA-Z0-9,]+", text.lower())

        cleaned = set()

        for token in tokens:
            token = token.strip(",.")
            if token and token not in stopwords:
                cleaned.add(token)

        return cleaned
'''

(PROJECT_DIR / "src/legal_doc_ai/hallucination_guard.py").write_text(
    hallucination_guard_code,
    encoding="utf-8"
)

print("hallucination_guard.py created.")

hallucination_guard.py created.


# **Creating Project Pipeline**

In [16]:
pipeline_code = r'''
from __future__ import annotations

import argparse
from pathlib import Path

from .chunking import TextChunker
from .config import Settings
from .document_processor import DocumentProcessor
from .edit_learner import EditLearner
from .evaluator import Evaluator
from .generator import GroundedDraftGenerator
from .hallucination_guard import HallucinationGuard
from .retriever import HybridRetriever
from .utils import ensure_dir, set_seed, write_json, write_jsonl


def run_pipeline(
    input_dir: str | Path,
    output_dir: str | Path,
    task: str,
    simulate_edit: bool = True,
    use_hallucination_guard: bool = True
) -> dict:
    settings = Settings()
    set_seed(settings.random_seed)

    input_dir = Path(input_dir)
    output_dir = ensure_dir(output_dir)

    processor = DocumentProcessor()

    pages, structured_fields = processor.process_directory(input_dir)

    write_jsonl(output_dir / "extracted_documents.jsonl", pages)

    write_json(
        output_dir / "structured_fields.json",
        [item.model_dump() for item in structured_fields]
    )

    chunker = TextChunker(
        chunk_size_chars=settings.chunk_size_chars,
        overlap_chars=settings.chunk_overlap_chars
    )

    chunks = chunker.chunk_pages(pages)

    write_jsonl(output_dir / "chunks.jsonl", chunks)

    retriever = HybridRetriever(settings.embedding_model)
    retriever.fit(chunks)

    evidence = retriever.search(task, top_k=settings.top_k)

    write_json(
        output_dir / "retrieval_trace.json",
        [item.model_dump() for item in evidence]
    )

    generator = GroundedDraftGenerator(
        backend=settings.generation_backend,
        openai_model=settings.openai_model
    )

    draft_result = generator.generate(
        task=task,
        evidence=evidence,
        structured_fields=structured_fields
    )

    raw_draft = draft_result.draft

    (output_dir / "draft_raw_generated.md").write_text(
        raw_draft,
        encoding="utf-8"
    )

    if use_hallucination_guard:
        guard = HallucinationGuard(
            min_support_score=0.22,
            min_claim_tokens=5
        )

        guarded_draft, hallucination_report = guard.guard_draft(
            draft=raw_draft,
            evidence=evidence,
            structured_fields=structured_fields
        )

        write_json(
            output_dir / "hallucination_report.json",
            hallucination_report
        )

        (output_dir / "draft_guarded.md").write_text(
            guarded_draft,
            encoding="utf-8"
        )

        default_draft = guarded_draft

    else:
        hallucination_report = {
            "guard_enabled": False
        }

        default_draft = raw_draft

    (output_dir / "draft_default.md").write_text(
        default_draft,
        encoding="utf-8"
    )

    edit_learner = EditLearner(output_dir / "edit_memory.json")

    if simulate_edit:
        edited_draft = default_draft.replace(
            "The records indicate possible notice-related issues involving unpaid rent, late payment, and written notice requirements.",
            "The records indicate possible nonpayment, late-payment issues, and written notice requirements."
        )

        edited_draft = edited_draft.replace(
            "before determining the appropriate next step.",
            "before determining the appropriate next step or legal remedy."
        )

        edit_learner.learn_from_edit(default_draft, edited_draft)
    else:
        edit_learner.save()

    improved_draft = edit_learner.apply(default_draft)

    if use_hallucination_guard:
        guard = HallucinationGuard(
            min_support_score=0.22,
            min_claim_tokens=5
        )

        improved_draft, improved_hallucination_report = guard.guard_draft(
            draft=improved_draft,
            evidence=evidence,
            structured_fields=structured_fields
        )

        write_json(
            output_dir / "hallucination_report_after_edit_learning.json",
            improved_hallucination_report
        )

    (output_dir / "draft_after_edit_learning.md").write_text(
        improved_draft,
        encoding="utf-8"
    )

    evaluator = Evaluator()

    report = evaluator.evaluate(
        evidence=evidence,
        draft=default_draft,
        improved_draft=improved_draft
    )

    report_dict = report.model_dump()

    report_dict["hallucination_guard_enabled"] = use_hallucination_guard

    if use_hallucination_guard:
        report_dict["hallucination_claims_checked"] = hallucination_report.get(
            "total_claims_checked",
            0
        )
        report_dict["hallucination_claims_removed"] = hallucination_report.get(
            "removed_or_quarantined_claims",
            0
        )

    write_json(output_dir / "evaluation_report.json", report_dict)

    return {
        "pages_processed": len(pages),
        "chunks_created": len(chunks),
        "evidence_items": len(evidence),
        "output_dir": str(output_dir),
        "hallucination_guard_enabled": use_hallucination_guard,
        "evaluation": report_dict
    }


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="Run legal document AI assessment pipeline."
    )

    parser.add_argument("--input_dir", type=str, default="data/sample_docs")
    parser.add_argument("--output_dir", type=str, default="outputs/sample_run")
    parser.add_argument(
        "--task",
        type=str,
        default="Create a notice-related case fact summary."
    )
    parser.add_argument("--simulate_edit", action="store_true")
    parser.add_argument("--disable_hallucination_guard", action="store_true")

    return parser


def main() -> None:
    parser = build_parser()
    args = parser.parse_args()

    summary = run_pipeline(
        input_dir=args.input_dir,
        output_dir=args.output_dir,
        task=args.task,
        simulate_edit=args.simulate_edit,
        use_hallucination_guard=not args.disable_hallucination_guard
    )

    print(summary)


if __name__ == "__main__":
    main()
'''

(PROJECT_DIR / "src/legal_doc_ai/pipeline.py").write_text(
    pipeline_code,
    encoding="utf-8"
)

print("pipeline.py with hallucination removal created.")

pipeline.py with hallucination removal created.


# **Project Path Setup**

In [17]:
import sys

SRC_DIR = str(PROJECT_DIR / "src")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("Python path ready.")

Python path ready.


# **Full system with Hallucination Removal**

In [18]:
from legal_doc_ai.pipeline import run_pipeline

INPUT_DIR = PROJECT_DIR / "data/sample_docs"
OUTPUT_DIR = PROJECT_DIR / "outputs/sample_run"

result = run_pipeline(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    task="Create a notice-related case fact summary.",
    simulate_edit=True,
    use_hallucination_guard=True
)

result

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

{'pages_processed': 3,
 'chunks_created': 3,
 'evidence_items': 3,
 'output_dir': '/content/legal_doc_ai_assessment/outputs/sample_run',
 'hallucination_guard_enabled': True,
 'evaluation': {'retrieval_recall_at_k': 1.0,
  'grounding_coverage': 1.0,
  'unsupported_claim_control': True,
  'edit_learning_applied': True,
  'notes': ['Synthetic gold facts are used for a small reviewer-friendly evaluation.',
   'Grounding coverage checks whether retrieved evidence IDs are visible in the draft.',
   'Unsupported claim control checks whether missing facts are explicitly marked unclear.',
   'Edit learning checks whether operator preferences are applied to later drafts.'],
  'hallucination_guard_enabled': True,
  'hallucination_claims_checked': 8,
  'hallucination_claims_removed': 0}}

# **Report1**

In [19]:
import json

hallucination_report = json.loads(
    (OUTPUT_DIR / "hallucination_report.json").read_text(encoding="utf-8")
)

print(json.dumps(hallucination_report, indent=2))

{
  "guard_enabled": true,
  "min_support_score": 0.22,
  "total_claims_checked": 8,
  "supported_claims": 8,
  "removed_or_quarantined_claims": 0,
  "removed_claims": [],
  "claim_checks": [
    {
      "claim": "The records indicate possible notice-related issues involving unpaid rent, late payment, and written notice requirements. This summary is limited to the retrieved source evidence and does not make a final legal conclusion.",
      "support_score": 0.391,
      "supported": true,
      "best_evidence_id": "operator_edit_example:p1:c0",
      "reason": "Claim has enough lexical overlap with retrieved evidence."
    },
    {
      "claim": "Landlord / property manager: Northgate Property Management LLC",
      "support_score": 0.8,
      "supported": true,
      "best_evidence_id": "notice_to_vacate:p1:c0",
      "reason": "Claim has enough lexical overlap with retrieved evidence."
    },
    {
      "claim": "Property: 44 West Pine Street, Unit 3B, Seattle, WA 98101",
      "su

# **Guarded Draft**

In [20]:
guarded_draft = (OUTPUT_DIR / "draft_guarded.md").read_text(encoding="utf-8")

print(guarded_draft)

# Notice-Related Case Fact Summary

## Drafting task

Create a notice-related case fact summary.

## Short answer

The records indicate possible notice-related issues involving unpaid rent, late payment, and written notice requirements. This summary is limited to the retrieved source evidence and does not make a final legal conclusion.

## Key extracted facts

- Tenant: Daniel Harper
- Landlord / property manager: Northgate Property Management LLC
- Property: 44 West Pine Street, Unit 3B, Seattle, WA 98101
- Notice date: March 12, 2026
- Lease start or lease date: January 1, 2025
- Monthly rent: $1,850
- Late fee: $1,850
- Stated balance: $1,925
- Main issues found: late payment, unpaid rent, written notice

## Source-grounded summary

The available records indicate that the tenant and landlord/property manager relationship concerns the property at 44 West Pine Street, Unit 3B, Seattle, WA 98101. The documents identify rent-related obligations, including monthly rent of $1,850 and a la

# **Final improved draft**

In [21]:
final_draft = (OUTPUT_DIR / "draft_after_edit_learning.md").read_text(encoding="utf-8")

print(final_draft)

# Notice-Related Case Fact Summary

## Drafting task

Create a notice-related case fact summary.

## Short answer

The records indicate possible notice-related issues involving unpaid rent, late payment, and written notice requirements. This summary is limited to the retrieved source evidence and does not make a final legal conclusion.

## Key extracted facts

- Tenant: Daniel Harper
- Landlord / property manager: Northgate Property Management LLC
- Property: 44 West Pine Street, Unit 3B, Seattle, WA 98101
- Notice date: March 12, 2026
- Lease start or lease date: January 1, 2025
- Monthly rent: $1,850
- Late fee: $1,850
- Stated balance: $1,925
- Main issues found: late payment, unpaid rent, written notice

## Source-grounded summary

The available records indicate that the tenant and landlord/property manager relationship concerns the property at 44 West Pine Street, Unit 3B, Seattle, WA 98101. The documents identify rent-related obligations, including monthly rent of $1,850 and a la

# **Final evaluation**

In [22]:
evaluation_report = json.loads(
    (OUTPUT_DIR / "evaluation_report.json").read_text(encoding="utf-8")
)

print(json.dumps(evaluation_report, indent=2))

{
  "retrieval_recall_at_k": 1.0,
  "grounding_coverage": 1.0,
  "unsupported_claim_control": true,
  "edit_learning_applied": true,
  "notes": [
    "Synthetic gold facts are used for a small reviewer-friendly evaluation.",
    "Grounding coverage checks whether retrieved evidence IDs are visible in the draft.",
    "Unsupported claim control checks whether missing facts are explicitly marked unclear.",
    "Edit learning checks whether operator preferences are applied to later drafts."
  ],
  "hallucination_guard_enabled": true,
  "hallucination_claims_checked": 8,
  "hallucination_claims_removed": 0
}


# **ReadMe File Created**

In [23]:
readme_text = '''
# Legal Document Understanding, Grounded Drafting, Hallucination Removal, and Edit Learning

This project is a Google Colab-first AI Engineer assessment solution.

It processes messy legal-style documents, extracts text and structured fields, retrieves relevant evidence, generates a grounded notice-related case fact summary, removes unsupported hallucinated claims, and improves future drafts from operator edits.

## Main workflow

1. Document processing

Supports TXT, PDF, and image files. Uses embedded PDF text when available. Falls back to OCR for scanned or noisy documents. Saves extracted text, OCR confidence, warnings, and structured fields.

2. Grounded retrieval

Splits documents into page-aware chunks. Uses hybrid retrieval with BM25 and sentence-transformer embeddings. Saves retrieval traces for inspection.

3. Draft generation

Generates a notice-related case fact summary. Uses retrieved evidence and structured fields. Marks unsupported or unclear points instead of guessing.

4. Hallucination removal

Checks material claims against retrieved evidence. Removes unsupported or high-risk legal claims. Saves hallucination_report.json. Produces draft_guarded.md.

5. Improvement from edits

Simulates an operator edit. Learns reusable preferences such as cautious legal wording. Applies those preferences to future drafts. Runs hallucination removal again after edit learning.

6. Evaluation

Measures retrieval recall against synthetic gold facts. Measures grounding coverage by checking evidence references. Checks unsupported claim control. Checks whether edit learning was applied. Reports hallucination guard activity.

## How to run in Google Colab

Run the notebook cells from top to bottom.

Main command:

from legal_doc_ai.pipeline import run_pipeline

run_pipeline(
    input_dir="data/sample_docs",
    output_dir="outputs/sample_run",
    task="Create a notice-related case fact summary.",
    simulate_edit=True,
    use_hallucination_guard=True
)

## Main output files

outputs/sample_run/extracted_documents.jsonl
outputs/sample_run/structured_fields.json
outputs/sample_run/chunks.jsonl
outputs/sample_run/retrieval_trace.json
outputs/sample_run/draft_raw_generated.md
outputs/sample_run/draft_guarded.md
outputs/sample_run/hallucination_report.json
outputs/sample_run/draft_default.md
outputs/sample_run/draft_after_edit_learning.md
outputs/sample_run/hallucination_report_after_edit_learning.json
outputs/sample_run/edit_memory.json
outputs/sample_run/evaluation_report.json

## Assumptions and tradeoffs

The sample documents are synthetic.

The goal is source-grounded drafting, not legal advice.

The deterministic template generator is used by default for reproducibility.

OCR confidence is preserved because messy scans may be partially unclear.

The hallucination guard uses transparent evidence-overlap scoring rather than another LLM judge.

The edit-learning loop stores reusable drafting preferences instead of doing model fine-tuning.

Hybrid retrieval is used because legal documents need both exact keyword matching and semantic search.

## Why hallucination removal matters

Legal-style drafting is risky if the system invents facts or legal conclusions. The hallucination guard removes unsupported material claims and high-risk legal conclusions unless the retrieved evidence strongly supports them. This keeps the draft grounded, inspectable, and safer for operator review.
'''

(PROJECT_DIR / "README.md").write_text(readme_text, encoding="utf-8")

print("Updated README.md created successfully.")

Updated README.md created successfully.


In [24]:
print((PROJECT_DIR / "README.md").read_text(encoding="utf-8")[:1000])


# Legal Document Understanding, Grounded Drafting, Hallucination Removal, and Edit Learning

This project is a Google Colab-first AI Engineer assessment solution.

It processes messy legal-style documents, extracts text and structured fields, retrieves relevant evidence, generates a grounded notice-related case fact summary, removes unsupported hallucinated claims, and improves future drafts from operator edits.

## Main workflow

1. Document processing

Supports TXT, PDF, and image files. Uses embedded PDF text when available. Falls back to OCR for scanned or noisy documents. Saves extracted text, OCR confidence, warnings, and structured fields.

2. Grounded retrieval

Splits documents into page-aware chunks. Uses hybrid retrieval with BM25 and sentence-transformer embeddings. Saves retrieval traces for inspection.

3. Draft generation

Generates a notice-related case fact summary. Uses retrieved evidence and structured fields. Marks unsupported or unclear points instead of guessing.

# **A Cross Fact Summary related Notices**

In [25]:
from legal_doc_ai.pipeline import run_pipeline

INPUT_DIR = PROJECT_DIR / "data/sample_docs"
OUTPUT_DIR = PROJECT_DIR / "outputs/sample_run"

result = run_pipeline(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    task="Create a notice-related case fact summary.",
    simulate_edit=True,
    use_hallucination_guard=True
)

result

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'pages_processed': 3,
 'chunks_created': 3,
 'evidence_items': 3,
 'output_dir': '/content/legal_doc_ai_assessment/outputs/sample_run',
 'hallucination_guard_enabled': True,
 'evaluation': {'retrieval_recall_at_k': 1.0,
  'grounding_coverage': 1.0,
  'unsupported_claim_control': True,
  'edit_learning_applied': True,
  'notes': ['Synthetic gold facts are used for a small reviewer-friendly evaluation.',
   'Grounding coverage checks whether retrieved evidence IDs are visible in the draft.',
   'Unsupported claim control checks whether missing facts are explicitly marked unclear.',
   'Edit learning checks whether operator preferences are applied to later drafts.'],
  'hallucination_guard_enabled': True,
  'hallucination_claims_checked': 8,
  'hallucination_claims_removed': 0}}

In [26]:
from pathlib import Path

required_files = [
    "extracted_documents.jsonl",
    "structured_fields.json",
    "chunks.jsonl",
    "retrieval_trace.json",
    "draft_raw_generated.md",
    "draft_guarded.md",
    "hallucination_report.json",
    "draft_default.md",
    "draft_after_edit_learning.md",
    "hallucination_report_after_edit_learning.json",
    "edit_memory.json",
    "evaluation_report.json",
]

for file_name in required_files:
    file_path = OUTPUT_DIR / file_name
    print(file_name, "Exist" if file_path.exists() else "Doesn't Exist")

extracted_documents.jsonl Exist
structured_fields.json Exist
chunks.jsonl Exist
retrieval_trace.json Exist
draft_raw_generated.md Exist
draft_guarded.md Exist
hallucination_report.json Exist
draft_default.md Exist
draft_after_edit_learning.md Exist
hallucination_report_after_edit_learning.json Exist
edit_memory.json Exist
evaluation_report.json Exist


In [27]:
architecture_text = '''
# Architecture Overview

## Goal

The goal is to build an inspectable AI workflow that turns messy legal-style documents into a grounded first-pass draft and improves future drafts from operator edits.

## Pipeline

Input documents
↓
DocumentProcessor
- Extracts embedded text from PDFs.
- Falls back to OCR for scanned pages and images.
- Normalizes extracted text.
- Extracts structured fields such as parties, dates, amounts, property address, and issues.

↓
TextChunker
- Creates stable page-aware chunks.
- Preserves document ID and page number for citation.

↓
HybridRetriever
- Uses BM25 for exact keyword matching.
- Uses sentence-transformer embeddings with FAISS for semantic retrieval.
- Merges and ranks evidence.
- Saves retrieval_trace.json so evidence can be inspected.

↓
GroundedDraftGenerator
- Uses structured fields and retrieved evidence.
- Generates a notice-related case fact summary.
- Marks unsupported points clearly.
- Keeps evidence IDs visible in the draft.

↓
HallucinationGuard
- Checks material claims against retrieved evidence.
- Removes unsupported or high-risk legal claims.
- Saves hallucination_report.json.
- Produces a safer guarded draft.

↓
EditLearner
- Captures default draft and edited draft.
- Extracts reusable edit preferences.
- Stores edit memory as JSON.
- Applies cautious language to future drafts.

↓
Evaluator
- Checks retrieval recall.
- Checks grounding coverage.
- Checks unsupported claim control.
- Checks whether edit learning was applied.
- Reports hallucination guard activity.

## Why this design is appropriate

The assessment focuses on document understanding, grounded drafting, and improvement from operator edits. This design makes every stage inspectable by saving intermediate artifacts.

## Key tradeoffs

Rule-based structured extraction is simple and transparent, but not as flexible as schema-constrained LLM extraction.

Template drafting is reproducible, but less natural than LLM-based drafting.

The hallucination guard uses transparent evidence-overlap scoring instead of another LLM judge, which makes it easier to inspect.

Edit learning uses reusable preferences instead of fine-tuning, which is safer and easier to evaluate for a take-home assessment.

Synthetic documents are used because the assessment allows mock data.
'''

(PROJECT_DIR / "architecture.md").write_text(architecture_text, encoding="utf-8")

print("architecture.md created.")

architecture.md created.


In [28]:
requirements_text = '''
pymupdf
pdf2image
pytesseract
pillow
opencv-python-headless
sentence-transformers
rank-bm25
faiss-cpu
pydantic
rapidfuzz
python-dotenv
openai
pytest
'''

(PROJECT_DIR / "requirements.txt").write_text(requirements_text.strip(), encoding="utf-8")

print("requirements.txt created.")

requirements.txt created.


# **A Visual Dashboard**

In [30]:
visualizer_code = r'''
from __future__ import annotations

import json
import html
from pathlib import Path
from typing import Any


def read_json_safe(path: Path, default: Any):
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return default


def read_text_safe(path: Path, default: str = "") -> str:
    if not path.exists():
        return default
    try:
        return path.read_text(encoding="utf-8")
    except Exception:
        return default


def percent(value: Any) -> str:
    try:
        value = float(value)
        if value <= 1:
            value = value * 100
        return f"{value:.1f}%"
    except Exception:
        return "N/A"


def bool_badge(value: Any) -> str:
    if value is True:
        return '<span class="badge good">Passed</span>'
    if value is False:
        return '<span class="badge bad">Failed</span>'
    return '<span class="badge neutral">N/A</span>'


def short_text(value: str, limit: int = 420) -> str:
    value = " ".join(str(value).split())
    if len(value) <= limit:
        return value
    return value[:limit].rstrip() + "..."


def create_visual_report(output_dir: str | Path, report_path: str | Path) -> Path:
    output_dir = Path(output_dir)
    report_path = Path(report_path)

    evaluation = read_json_safe(output_dir / "evaluation_report.json", {})
    hallucination = read_json_safe(output_dir / "hallucination_report.json", {})
    hallucination_after = read_json_safe(
        output_dir / "hallucination_report_after_edit_learning.json",
        {}
    )
    retrieval_trace = read_json_safe(output_dir / "retrieval_trace.json", [])
    structured_fields = read_json_safe(output_dir / "structured_fields.json", [])

    guarded_draft = read_text_safe(output_dir / "draft_guarded.md", "")
    final_draft = read_text_safe(output_dir / "draft_after_edit_learning.md", "")

    recall = evaluation.get("retrieval_recall_at_k", "N/A")
    grounding = evaluation.get("grounding_coverage", "N/A")
    unsupported_control = evaluation.get("unsupported_claim_control")
    edit_learning = evaluation.get("edit_learning_applied")

    claims_checked = hallucination.get("total_claims_checked", 0)
    claims_removed = hallucination.get("removed_or_quarantined_claims", 0)

    after_claims_checked = hallucination_after.get("total_claims_checked", 0)
    after_claims_removed = hallucination_after.get("removed_or_quarantined_claims", 0)

    evidence_cards = ""

    for item in retrieval_trace:
        chunk_id = html.escape(str(item.get("chunk_id", "unknown")))
        source = html.escape(str(item.get("source_path", "unknown")))
        page = html.escape(str(item.get("page_number", "N/A")))
        method = html.escape(str(item.get("retrieval_method", "N/A")))
        score = item.get("score", 0)
        text = html.escape(short_text(item.get("text", "")))

        evidence_cards += f"""
        <div class="evidence-card">
            <div class="evidence-top">
                <span class="rank">Rank {item.get("rank", "N/A")}</span>
                <span class="chip">{method}</span>
                <span class="score">Score: {float(score):.3f}</span>
            </div>
            <div class="chunk-id">{chunk_id}</div>
            <div class="source">Source: {source} | Page: {page}</div>
            <p>{text}</p>
        </div>
        """

    if not evidence_cards:
        evidence_cards = '<p class="muted">No retrieval evidence found.</p>'

    structured_html = ""

    for doc in structured_fields:
        doc_id = html.escape(str(doc.get("doc_id", "unknown")))

        parties = doc.get("parties", {})
        dates = doc.get("dates", {})
        amounts = doc.get("amounts", {})
        property_address = html.escape(str(doc.get("property_address", "N/A")))
        issues = ", ".join(doc.get("issues", [])) or "N/A"
        issues = html.escape(issues)

        structured_html += f"""
        <div class="structured-card">
            <h3>{doc_id}</h3>
            <table>
                <tr><th>Property</th><td>{property_address}</td></tr>
                <tr><th>Parties</th><td><pre>{html.escape(json.dumps(parties, indent=2))}</pre></td></tr>
                <tr><th>Dates</th><td><pre>{html.escape(json.dumps(dates, indent=2))}</pre></td></tr>
                <tr><th>Amounts</th><td><pre>{html.escape(json.dumps(amounts, indent=2))}</pre></td></tr>
                <tr><th>Issues</th><td>{issues}</td></tr>
            </table>
        </div>
        """

    if not structured_html:
        structured_html = '<p class="muted">No structured fields found.</p>'

    removed_claims = hallucination.get("removed_claims", [])
    removed_claims_html = ""

    for item in removed_claims:
        claim = html.escape(str(item.get("claim", "")))
        reason = html.escape(str(item.get("reason", "")))
        support_score = item.get("support_score", "N/A")

        removed_claims_html += f"""
        <div class="removed-claim">
            <p><strong>Removed claim:</strong> {claim}</p>
            <p><strong>Reason:</strong> {reason}</p>
            <p><strong>Support score:</strong> {support_score}</p>
        </div>
        """

    if not removed_claims_html:
        removed_claims_html = '<p class="muted">No unsupported claims were removed in the first guarded draft.</p>'

    html_report = f"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Legal Document AI Assessment Report</title>
<style>
    body {{
        margin: 0;
        background: #f4f7fb;
        color: #172033;
        font-family: Arial, Helvetica, sans-serif;
    }}

    .page {{
        max-width: 1180px;
        margin: 0 auto;
        padding: 34px 22px 60px;
    }}

    .hero {{
        background: linear-gradient(135deg, #101828, #1d4ed8);
        color: white;
        padding: 34px;
        border-radius: 24px;
        box-shadow: 0 20px 50px rgba(16, 24, 40, 0.18);
        margin-bottom: 28px;
    }}

    .hero h1 {{
        margin: 0 0 10px;
        font-size: 34px;
        letter-spacing: -0.7px;
    }}

    .hero p {{
        margin: 0;
        max-width: 860px;
        color: #dbeafe;
        font-size: 16px;
        line-height: 1.6;
    }}

    .grid {{
        display: grid;
        grid-template-columns: repeat(4, 1fr);
        gap: 16px;
        margin-bottom: 28px;
    }}

    .metric-card {{
        background: white;
        padding: 22px;
        border-radius: 18px;
        box-shadow: 0 10px 30px rgba(16, 24, 40, 0.08);
        border: 1px solid #e7eef8;
    }}

    .metric-label {{
        color: #667085;
        font-size: 13px;
        margin-bottom: 8px;
    }}

    .metric-value {{
        font-size: 28px;
        font-weight: 800;
        color: #101828;
    }}

    .section {{
        background: white;
        padding: 26px;
        border-radius: 22px;
        box-shadow: 0 10px 30px rgba(16, 24, 40, 0.07);
        border: 1px solid #e7eef8;
        margin-bottom: 24px;
    }}

    .section h2 {{
        margin: 0 0 16px;
        font-size: 23px;
        color: #101828;
    }}

    .pipeline {{
        display: grid;
        grid-template-columns: repeat(6, 1fr);
        gap: 12px;
    }}

    .stage {{
        background: #f8fbff;
        border: 1px solid #dbeafe;
        border-radius: 16px;
        padding: 16px;
        min-height: 120px;
    }}

    .stage-number {{
        width: 28px;
        height: 28px;
        border-radius: 999px;
        background: #2563eb;
        color: white;
        display: inline-flex;
        align-items: center;
        justify-content: center;
        font-weight: 700;
        margin-bottom: 10px;
    }}

    .stage h3 {{
        margin: 0 0 8px;
        font-size: 15px;
    }}

    .stage p {{
        margin: 0;
        color: #667085;
        font-size: 13px;
        line-height: 1.45;
    }}

    .badge {{
        display: inline-block;
        padding: 7px 11px;
        border-radius: 999px;
        font-size: 13px;
        font-weight: 700;
    }}

    .good {{
        background: #dcfce7;
        color: #166534;
    }}

    .bad {{
        background: #fee2e2;
        color: #991b1b;
    }}

    .neutral {{
        background: #e5e7eb;
        color: #374151;
    }}

    .evidence-card, .structured-card, .removed-claim {{
        background: #f8fbff;
        border: 1px solid #dbeafe;
        border-radius: 16px;
        padding: 18px;
        margin-bottom: 14px;
    }}

    .evidence-top {{
        display: flex;
        gap: 10px;
        align-items: center;
        flex-wrap: wrap;
        margin-bottom: 10px;
    }}

    .rank {{
        font-weight: 800;
        color: #1d4ed8;
    }}

    .chip {{
        background: #e0f2fe;
        color: #075985;
        padding: 5px 9px;
        border-radius: 999px;
        font-size: 12px;
        font-weight: 700;
    }}

    .score {{
        color: #667085;
        font-size: 13px;
    }}

    .chunk-id {{
        font-family: Consolas, monospace;
        font-size: 13px;
        color: #344054;
        margin-bottom: 6px;
    }}

    .source {{
        color: #667085;
        font-size: 13px;
        margin-bottom: 8px;
    }}

    .evidence-card p {{
        margin: 0;
        line-height: 1.6;
        color: #344054;
    }}

    table {{
        width: 100%;
        border-collapse: collapse;
        margin-top: 10px;
    }}

    th, td {{
        border-bottom: 1px solid #e5e7eb;
        padding: 12px;
        text-align: left;
        vertical-align: top;
    }}

    th {{
        width: 180px;
        color: #475467;
        background: #f9fafb;
    }}

    pre {{
        white-space: pre-wrap;
        margin: 0;
        font-family: Consolas, monospace;
        font-size: 13px;
    }}

    .draft-box {{
        background: #0b1220;
        color: #e5e7eb;
        padding: 22px;
        border-radius: 16px;
        overflow-x: auto;
        white-space: pre-wrap;
        line-height: 1.6;
        font-family: Consolas, monospace;
        font-size: 13px;
        max-height: 560px;
        overflow-y: auto;
    }}

    .muted {{
        color: #667085;
    }}

    .two-col {{
        display: grid;
        grid-template-columns: 1fr 1fr;
        gap: 16px;
    }}

    .footer {{
        text-align: center;
        color: #667085;
        font-size: 13px;
        margin-top: 28px;
    }}

    @media (max-width: 950px) {{
        .grid {{
            grid-template-columns: repeat(2, 1fr);
        }}
        .pipeline {{
            grid-template-columns: repeat(2, 1fr);
        }}
        .two-col {{
            grid-template-columns: 1fr;
        }}
    }}

    @media (max-width: 600px) {{
        .grid {{
            grid-template-columns: 1fr;
        }}
        .pipeline {{
            grid-template-columns: 1fr;
        }}
    }}
</style>
</head>
<body>
<div class="page">

    <div class="hero">
        <h1>Legal Document AI Assessment Report</h1>
        <p>
            A complete visual summary of the document-processing, grounded retrieval,
            hallucination-removal, draft-generation, edit-learning, and evaluation workflow.
            This report is generated directly from the pipeline outputs.
        </p>
    </div>

    <div class="grid">
        <div class="metric-card">
            <div class="metric-label">Retrieval Recall@K</div>
            <div class="metric-value">{percent(recall)}</div>
        </div>
        <div class="metric-card">
            <div class="metric-label">Grounding Coverage</div>
            <div class="metric-value">{percent(grounding)}</div>
        </div>
        <div class="metric-card">
            <div class="metric-label">Unsupported Claim Control</div>
            <div class="metric-value">{bool_badge(unsupported_control)}</div>
        </div>
        <div class="metric-card">
            <div class="metric-label">Edit Learning Applied</div>
            <div class="metric-value">{bool_badge(edit_learning)}</div>
        </div>
    </div>

    <div class="section">
        <h2>Pipeline Overview</h2>
        <div class="pipeline">
            <div class="stage">
                <div class="stage-number">1</div>
                <h3>Document Processing</h3>
                <p>Extracts embedded text, runs OCR fallback, normalizes text, and captures confidence warnings.</p>
            </div>
            <div class="stage">
                <div class="stage-number">2</div>
                <h3>Structured Extraction</h3>
                <p>Pulls parties, dates, amounts, property address, issues, and uncertainty notes.</p>
            </div>
            <div class="stage">
                <div class="stage-number">3</div>
                <h3>Hybrid Retrieval</h3>
                <p>Combines BM25 keyword search with FAISS semantic search to retrieve grounded evidence.</p>
            </div>
            <div class="stage">
                <div class="stage-number">4</div>
                <h3>Draft Generation</h3>
                <p>Creates a first-pass notice-related case fact summary using only retrieved evidence.</p>
            </div>
            <div class="stage">
                <div class="stage-number">5</div>
                <h3>Hallucination Guard</h3>
                <p>Checks claims against evidence and removes unsupported or high-risk legal claims.</p>
            </div>
            <div class="stage">
                <div class="stage-number">6</div>
                <h3>Edit Learning</h3>
                <p>Learns cautious drafting preferences from operator edits and applies them to future drafts.</p>
            </div>
        </div>
    </div>

    <div class="section">
        <h2>Hallucination Removal Summary</h2>
        <div class="grid">
            <div class="metric-card">
                <div class="metric-label">Claims Checked Before Edit Learning</div>
                <div class="metric-value">{claims_checked}</div>
            </div>
            <div class="metric-card">
                <div class="metric-label">Claims Removed Before Edit Learning</div>
                <div class="metric-value">{claims_removed}</div>
            </div>
            <div class="metric-card">
                <div class="metric-label">Claims Checked After Edit Learning</div>
                <div class="metric-value">{after_claims_checked}</div>
            </div>
            <div class="metric-card">
                <div class="metric-label">Claims Removed After Edit Learning</div>
                <div class="metric-value">{after_claims_removed}</div>
            </div>
        </div>
        <h3>Removed or Quarantined Claims</h3>
        {removed_claims_html}
    </div>

    <div class="section">
        <h2>Structured Fields Extracted</h2>
        {structured_html}
    </div>

    <div class="section">
        <h2>Retrieved Evidence</h2>
        {evidence_cards}
    </div>

    <div class="section">
        <h2>Final Draft After Hallucination Removal and Edit Learning</h2>
        <div class="draft-box">{html.escape(final_draft or guarded_draft or "No draft found.")}</div>
    </div>

    <div class="footer">
        Generated from pipeline artifacts in outputs/sample_run.
    </div>

</div>
</body>
</html>
"""

    report_path.write_text(html_report, encoding="utf-8")
    return report_path
'''

(PROJECT_DIR / "src/legal_doc_ai/visualizer.py").write_text(
    visualizer_code,
    encoding="utf-8"
)

print("visualizer.py created.")

visualizer.py created.


In [31]:
import sys

SRC_DIR = str(PROJECT_DIR / "src")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from legal_doc_ai.visualizer import create_visual_report

visual_report_path = create_visual_report(
    output_dir=PROJECT_DIR / "outputs/sample_run",
    report_path=PROJECT_DIR / "visual_report.html"
)

print("Visual report created at:", visual_report_path)

Visual report created at: /content/legal_doc_ai_assessment/visual_report.html


In [32]:
from IPython.display import HTML, display

display(HTML((PROJECT_DIR / "visual_report.html").read_text(encoding="utf-8")))

Property,"44 West Pine Street, Unit 3B, Seattle, WA 98101"
Parties,"{ ""landlord"": ""Northgate Property Management LLC"", ""tenant"": ""Daniel Harper"" }"
Dates,"{ ""lease_start"": ""January 1, 2025"", ""notice_date"": ""1st day of each month"" }"
Amounts,"{ ""monthly_rent"": ""$1,850"", ""late_fee"": ""$75"" }"
Issues,"late payment, written notice"
Property,"44 West Pine Street, Unit 3B, Seattle, WA 98101"
Parties,"{ ""landlord"": ""Northgate Property Management LLC"", ""tenant"": ""Daniel Harper"" }"
Dates,"{ ""notice_date"": ""March 12, 2026"" }"
Amounts,"{ ""late_fee"": ""$1,850"", ""balance"": ""$1,925"" }"
Issues,"unpaid rent, late payment, written notice"
Property,None


In [33]:
import shutil

shutil.copy(
    PROJECT_DIR / "visual_report.html",
    PROJECT_DIR / "outputs/sample_run/visual_report.html"
)

print("Copied to outputs/sample_run/visual_report.html")

Copied to outputs/sample_run/visual_report.html


In [34]:
from google.colab import files

files.download(str(PROJECT_DIR / "visual_report.html"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
readme_path = PROJECT_DIR / "README.md"

existing_readme = readme_path.read_text(encoding="utf-8")

visual_note = '''

## Visual Report

The project includes a polished HTML visual report:

visual_report.html

It summarizes:
- pipeline stages
- retrieval and grounding metrics
- hallucination-removal results
- structured extracted fields
- retrieved evidence
- final draft after edit learning

The same report is also saved at:

outputs/sample_run/visual_report.html
'''

if "## Visual Report" not in existing_readme:
    readme_path.write_text(existing_readme + visual_note, encoding="utf-8")

print("README updated with visual report section.")

README updated with visual report section.


In [37]:
files_to_check = [
    PROJECT_DIR / "visual_report.html",
    PROJECT_DIR / "outputs/sample_run/visual_report.html",
    PROJECT_DIR / "src/legal_doc_ai/visualizer.py",
]

for file_path in files_to_check:
    print(file_path.name, "Exist" if file_path.exists() else "Doesn't Exist")

visual_report.html Exist
visual_report.html Exist
visualizer.py Exist


In [38]:
import shutil

zip_path = shutil.make_archive(
    base_name="/content/legal_doc_ai_assessment",
    format="zip",
    root_dir="/content",
    base_dir="legal_doc_ai_assessment"
)

print(zip_path)

/content/legal_doc_ai_assessment.zip


In [39]:
from google.colab import files

files.download("/content/legal_doc_ai_assessment.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>